# 02 — Silver Layer Cleaning, Standardization, Data Quality & Quarantine
- **Workstream 2:** Lakehouse & Data Quality (Member 2 — Ahmed Essam)
- **ClickUp Tasks:**
  - `123u8tync4v` (*Implement Silver cleaning and standardization*)
  - `123u8tync4x` (*Create data-quality checks and report*)
- **Key Rule Enforced:** Quarantine bad/conflicting records with explicit reason codes (`silver_quarantine_records`); never silently drop records.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_ndc = spark.table("workspace.default.bronze_openfda_ndc")
bronze_who = spark.table("workspace.default.bronze_who_eml")

# Build single-row lowercase WHO EML text corpus for active-ingredient cross-matching
who_corpus_df = bronze_who.agg(
    F.concat_ws(" ", F.collect_list(F.lower(F.coalesce(F.col("text_preview"), F.lit(""))))).alias("who_corpus_lower")
)

# 1. Deduplication & validation rules over openFDA NDC
w_pk = Window.partitionBy("product_id").orderBy(F.col("ingestion_timestamp_utc").desc())
staged_ndc = (
    bronze_ndc
    .withColumn("row_num", F.row_number().over(w_pk))
    .withColumn(
        "has_active_ingredients",
        F.expr(
            "active_ingredients IS NOT NULL AND size(active_ingredients) > 0 "
            "AND active_ingredients[0].name IS NOT NULL AND trim(active_ingredients[0].name) != '' "
            "AND active_ingredients[0].strength IS NOT NULL AND trim(active_ingredients[0].strength) != ''"
        )
    )
    .withColumn("has_primary_key", F.expr("product_id IS NOT NULL AND trim(product_id) != '' AND product_ndc IS NOT NULL"))
    .withColumn("has_drug_name", F.expr("(brand_name IS NOT NULL AND trim(brand_name) != '') OR (generic_name IS NOT NULL AND trim(generic_name) != '')"))
)

# 2. Split NDC into Valid Silver vs Quarantine with explicit reason codes and details
quarantine_ndc = (
    staged_ndc.filter("NOT (row_num = 1 AND has_active_ingredients AND has_primary_key AND has_drug_name)")
    .select(
        F.concat(F.lit("Q-NDC-"), F.lpad(F.row_number().over(Window.orderBy("product_id")).cast("string"), 4, "0")).alias("quarantine_id"),
        F.lit("bronze_openfda_ndc").alias("source_table"),
        F.coalesce(F.col("product_id"), F.col("product_ndc"), F.lit("UNKNOWN")).alias("record_identifier"),
        F.when(~F.col("has_primary_key"), F.lit("MISSING_PRIMARY_IDENTIFIER"))
         .when(F.col("row_num") > 1, F.lit("DUPLICATE_PRIMARY_KEY"))
         .when(~F.col("has_drug_name"), F.lit("MISSING_DRUG_NAME"))
         .otherwise(F.lit("MISSING_ACTIVE_INGREDIENT_OR_STRENGTH")).alias("quarantine_reason_code"),
        F.when(~F.col("has_primary_key"), F.lit("Record lacks mandatory product_id or product_ndc."))
         .when(F.col("row_num") > 1, F.lit("Duplicate product_id detected during Silver deduplication."))
         .when(~F.col("has_drug_name"), F.lit("Record lacks both brand_name and generic_name."))
         .otherwise(F.lit("Record lacks structured active_ingredients array with name and strength.")).alias("quarantine_reason_detail"),
        F.lit("QUARANTINED_CRITICAL").alias("severity"),
        F.to_json(F.struct("*")).alias("raw_payload_json"),
        F.current_timestamp().alias("quarantined_at_utc")
    )
)

silver_ndc = (
    staged_ndc.filter("row_num = 1 AND has_active_ingredients AND has_primary_key AND has_drug_name")
    .crossJoin(F.broadcast(who_corpus_df))
    .select(
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("product_ndc")).alias("product_ndc"),
        F.upper(F.trim(F.coalesce(F.col("brand_name"), F.col("generic_name")))).alias("brand_name"),
        F.upper(F.trim(F.coalesce(F.col("generic_name"), F.col("brand_name")))).alias("generic_name"),
        F.upper(F.trim(F.coalesce(F.col("labeler_name"), F.lit("UNKNOWN LABELER")))).alias("labeler_name"),
        F.upper(F.trim(F.col("active_ingredients")[0]["name"])).alias("primary_active_ingredient"),
        F.trim(F.col("active_ingredients")[0]["strength"]).alias("primary_strength"),
        F.array_join(
            F.transform(
                F.col("active_ingredients"),
                lambda x: F.concat(F.upper(F.trim(x["name"])), F.lit(" ("), F.trim(x["strength"]), F.lit(")"))
            ),
            "; "
        ).alias("active_ingredients_all"),
        F.size(F.col("active_ingredients")).cast("int").alias("ingredient_count"),
        F.upper(F.trim(F.coalesce(F.col("dosage_form"), F.lit("UNKNOWN")))).alias("dosage_form"),
        F.upper(F.coalesce(F.array_join(F.col("route"), ", "), F.lit("UNSPECIFIED"))).alias("route"),
        F.upper(F.trim(F.coalesce(F.col("product_type"), F.lit("UNKNOWN")))).alias("product_type"),
        F.upper(F.trim(F.coalesce(F.col("marketing_category"), F.lit("UNSPECIFIED")))).alias("marketing_category"),
        F.coalesce(F.size(F.col("packaging")), F.lit(0)).cast("int").alias("package_count"),
        F.when(F.size(F.col("packaging")) > 0, F.col("packaging")[0]["description"]).alias("sample_package_description"),
        F.try_to_date(F.col("marketing_start_date"), "yyyyMMdd").alias("marketing_start_date_iso"),
        F.try_to_date(F.col("listing_expiration_date"), "yyyyMMdd").alias("listing_expiration_date_iso"),
        (
            (F.instr(F.col("who_corpus_lower"), F.lower(F.trim(F.col("active_ingredients")[0]["name"]))) > 0)
            | (
                (F.length(F.split(F.lower(F.trim(F.col("active_ingredients")[0]["name"])), " ")[0]) >= 5)
                & (F.instr(F.col("who_corpus_lower"), F.split(F.lower(F.trim(F.col("active_ingredients")[0]["name"])), " ")[0]) > 0)
            )
        ).alias("is_who_essential_ingredient"),
        F.col("source_id"),
        F.col("raw_record_sha256")
    )
)

(
    silver_ndc.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_openfda_ndc")
)
print(f"Silver NDC valid rows: {silver_ndc.count()} | Quarantined NDC rows: {quarantine_ndc.count()}")

In [ ]:
# 3. Clean & Standardize openFDA Enforcement Recalls (Zero-Silent-Drop Quarantine Enforced)
bronze_enf = spark.table("workspace.default.bronze_openfda_enforcement")
w_enf = Window.partitionBy("recall_number").orderBy(F.col("ingestion_timestamp_utc").desc())

staged_enf = (
    bronze_enf
    .withColumn("row_num", F.row_number().over(w_enf))
    .withColumn("has_recall_number", F.expr("recall_number IS NOT NULL AND trim(recall_number) != ''"))
    .withColumn("has_reason_and_product", F.expr("reason_for_recall IS NOT NULL AND trim(reason_for_recall) != '' AND product_description IS NOT NULL AND trim(product_description) != ''"))
)

quarantine_enf = (
    staged_enf.filter("NOT (row_num = 1 AND has_recall_number AND has_reason_and_product)")
    .select(
        F.concat(F.lit("Q-ENF-"), F.lpad(F.row_number().over(Window.orderBy("recall_number")).cast("string"), 4, "0")).alias("quarantine_id"),
        F.lit("bronze_openfda_enforcement").alias("source_table"),
        F.coalesce(F.col("recall_number"), F.col("event_id"), F.lit("UNKNOWN")).alias("record_identifier"),
        F.when(~F.col("has_recall_number"), F.lit("MISSING_RECALL_NUMBER"))
         .when(F.col("row_num") > 1, F.lit("DUPLICATE_RECALL_NUMBER"))
         .otherwise(F.lit("MISSING_REASON_OR_PRODUCT")).alias("quarantine_reason_code"),
        F.when(~F.col("has_recall_number"), F.lit("Enforcement record lacks mandatory recall_number."))
         .when(F.col("row_num") > 1, F.lit("Duplicate recall_number detected during Silver deduplication."))
         .otherwise(F.lit("Enforcement record lacks reason_for_recall or product_description.")).alias("quarantine_reason_detail"),
        F.lit("QUARANTINED_CRITICAL").alias("severity"),
        F.to_json(F.struct("*")).alias("raw_payload_json"),
        F.current_timestamp().alias("quarantined_at_utc")
    )
)

silver_enf = (
    staged_enf.filter("row_num = 1 AND has_recall_number AND has_reason_and_product")
    .withColumn("recall_initiation_date_iso", F.try_to_date(F.col("recall_initiation_date"), "yyyyMMdd"))
    .withColumn("report_date_iso", F.try_to_date(F.col("report_date"), "yyyyMMdd"))
    .select(
        F.trim(F.col("recall_number")).alias("recall_number"),
        F.col("event_id"),
        F.coalesce(F.col("classification"), F.lit("Unclassified")).alias("classification"),
        F.when(F.col("classification") == "Class I", 1)
         .when(F.col("classification") == "Class II", 2)
         .when(F.col("classification") == "Class III", 3)
         .otherwise(4).cast("int").alias("severity_rank"),
        F.initcap(F.coalesce(F.col("status"), F.lit("Unknown"))).alias("status"),
        F.upper(F.trim(F.coalesce(F.col("recalling_firm"), F.lit("UNKNOWN FIRM")))).alias("recalling_firm"),
        F.col("city"),
        F.col("state"),
        F.upper(F.trim(F.coalesce(F.col("country"), F.lit("UNITED STATES")))).alias("country"),
        F.when(F.lower(F.col("reason_for_recall")).rlike("steril"), "STERILITY_FAILURE")
         .when(F.lower(F.col("reason_for_recall")).rlike("microbial|contaminat|methanol|impurity|ndma|particulate"), "CONTAMINATION_OR_IMPURITY")
         .when(F.lower(F.col("reason_for_recall")).rlike("cgmp|good manufacturing"), "CGMP_DEVIATION")
         .when(F.lower(F.col("reason_for_recall")).rlike("unapproved|new drug"), "UNAPPROVED_DRUG_MARKETING")
         .when(F.lower(F.col("reason_for_recall")).rlike("label|misbrand|packaging"), "LABELING_OR_PACKAGING_DEFECT")
         .when(F.lower(F.col("reason_for_recall")).rlike("subpotent|superpotent|potency|dissolution|stability"), "POTENCY_OR_STABILITY_FAILURE")
         .otherwise("OTHER_QUALITY_DEFECT").alias("recall_reason_category"),
        F.col("reason_for_recall"),
        F.col("product_description"),
        F.col("recall_initiation_date_iso"),
        F.col("report_date_iso"),
        F.year(F.col("recall_initiation_date_iso")).cast("int").alias("recall_year"),
        F.col("source_id"),
        F.col("raw_record_sha256")
    )
)

all_quarantine = quarantine_ndc.unionByName(quarantine_enf)

(
    silver_enf.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_openfda_enforcement")
)
(
    all_quarantine.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_quarantine_records")
)

print(f"Silver Enforcement valid rows: {silver_enf.count()} | Total Quarantine rows: {all_quarantine.count()}")
display(all_quarantine)
display(silver_enf.limit(5))